In [2]:
import yfinance as yf
import pandas as pd

# European indices tickers on Yahoo Finance
indices = {
    "CAC40": "^FCHI",
    "DAX": "^GDAXI",
    "FTSE100": "^FTSE",
    "EuroStoxx50": "^STOXX50E"
}

# Download last 2 years of data
data = yf.download(
    tickers=list(indices.values()),
    period="2y",
    interval="1d",
    auto_adjust=True
)

# Keep only closing prices
close = data["Close"]
close.columns = list(indices.keys())

print(close.shape)
print(close.head())

[*********************100%***********************]  4 of 4 completed

(514, 4)
                  CAC40          DAX       FTSE100  EuroStoxx50
2024-05-22          NaN  8370.299805           NaN  5025.169922
2024-05-23          NaN  8339.200195           NaN  5037.600098
2024-05-24          NaN  8317.599609           NaN  5035.410156
2024-05-27  8132.490234          NaN  18774.710938  5059.200195
2024-05-28  8057.799805  8254.200195  18677.869141  5030.350098


In [3]:
# Check missing values percentage per index
missing = (close.isnull().sum() / len(close) * 100).round(2)
print("Missing values (%):")
print(missing)

# Check date range
print(f"\nDate range: {close.index[0].date()} → {close.index[-1].date()}")
print(f"Total trading days: {len(close)}")

Missing values (%):
CAC40          0.97
DAX            1.36
FTSE100        1.75
EuroStoxx50    2.53
dtype: float64

Date range: 2024-05-22 → 2026-05-25
Total trading days: 514


In [4]:
# ECB main interest rate data via yfinance
# EURIBOR 3 months as ECB rate proxy
euribor = yf.download("EUR=X", period="2y", interval="1d", auto_adjust=True)

# Also get EUR/USD exchange rate
eurusd = yf.download("EURUSD=X", period="2y", interval="1d", auto_adjust=True)

print("EURIBOR data:")
print(euribor["Close"].head())

print("\nEUR/USD data:")
print(eurusd["Close"].head())

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


EURIBOR data:
Ticker        EUR=X
2024-05-27  0.92186
2024-05-28  0.92065
2024-05-29  0.92150
2024-05-30  0.92577
2024-05-31  0.92290

EUR/USD data:
Ticker      EURUSD=X
2024-05-27  1.084763
2024-05-28  1.086189
2024-05-29  1.085187
2024-05-30  1.080182
2024-05-31  1.083541


In [12]:
pip install fredapi

  Using cached fredapi-0.5.2-py3-none-any.whl.metadata (5.0 kB)
Using cached fredapi-0.5.2-py3-none-any.whl (11 kB)
Note: you may need to restart the kernel to use updated packages.


In [21]:
from fredapi import Fred

fred = Fred(api_key=FRED_API_KEY)

# Fetch ECB deposit facility rate
ecb_rate = fred.get_series("ECBDFR", observation_start="2024-01-01")
print("ECB Rate:")
print(ecb_rate.tail(10))

# Alternative HICP series - Euro area annual inflation rate
inflation = fred.get_series("CPHPTT01EZM659N")
print(inflation.tail(10))
print(f"Total points: {len(inflation)}")

ECB Rate:
2026-05-13    2.0
2026-05-14    2.0
2026-05-15    2.0
2026-05-16    2.0
2026-05-17    2.0
2026-05-18    2.0
2026-05-19    2.0
2026-05-20    2.0
2026-05-21    2.0
2026-05-22    2.0
dtype: float64
2022-04-01     7.5
2022-05-01     8.1
2022-06-01     8.7
2022-07-01     8.9
2022-08-01     9.2
2022-09-01     9.9
2022-10-01    10.6
2022-11-01    10.1
2022-12-01     9.2
2023-01-01     8.7
dtype: float64
Total points: 385


In [24]:
series_to_try = [
    "CPALTT01EZM657N",
    "CPALTT01EZM659N", 
    "HICP0000EZ17M086NEST"
]

for series_id in series_to_try:
    try:
        data = fred.get_series(series_id, observation_start="2024-01-01")
        print(f"{series_id}: {len(data)} points")
        print(data.tail(3))
        print()
    except Exception as e:
        print(f"{series_id}: Error - {e}\n")

CPALTT01EZM657N: Error - Bad Request.  The series does not exist.

CPALTT01EZM659N: Error - Bad Request.  The series does not exist.

HICP0000EZ17M086NEST: Error - Bad Request.  The series does not exist.



In [27]:
import eurostat

# HICP monthly inflation - euro area
hicp = eurostat.get_data_df("prc_hicp_manr")
print(hicp.head())
print(hicp.shape)

  freq   unit coicop geo\TIME_PERIOD  1997-01  1997-02  1997-03  1997-04  \
0    M  RCH_A     AP              AL      NaN      NaN      NaN      NaN   
1    M  RCH_A     AP              AT      NaN      NaN      NaN      NaN   
2    M  RCH_A     AP              BE      NaN      NaN      NaN      NaN   
3    M  RCH_A     AP              BG      NaN      NaN      NaN      NaN   
4    M  RCH_A     AP              CH      NaN      NaN      NaN      NaN   

   1997-05  1997-06  ...  2025-03  2025-04  2025-05  2025-06  2025-07  \
0      NaN      NaN  ...      0.2      0.4      0.5      0.2     -0.2   
1      NaN      NaN  ...      5.0      5.0      5.0      4.8      5.4   
2      NaN      NaN  ...      4.1      4.3      4.3      4.3      4.3   
3      NaN      NaN  ...      7.1      0.1     -0.5     -0.2      0.2   
4      NaN      NaN  ...     -0.3     -0.2     -0.1     -0.1     -0.1   

   2025-08  2025-09  2025-10  2025-11  2025-12  
0     -0.2      0.2     -0.1     -0.3     -0.3  
1     

In [28]:
# Filter for Euro area (EA20) and all items (AP = all products)
hicp_ea = hicp[
    (hicp["geo\\TIME_PERIOD"] == "EA20") & 
    (hicp["coicop"] == "AP") &
    (hicp["unit"] == "RCH_A")
].copy()

# Keep only date columns
date_cols = [col for col in hicp_ea.columns if col.startswith("20")]
hicp_ea = hicp_ea[date_cols].T
hicp_ea.columns = ["inflation"]
hicp_ea.index = pd.to_datetime(hicp_ea.index)
hicp_ea = hicp_ea["2024-01-01":]

print(hicp_ea)

            inflation
2024-01-01        1.9
2024-02-01        2.5
2024-03-01        2.5
2024-04-01        2.1
2024-05-01        2.8
2024-06-01        3.4
2024-07-01        4.1
2024-08-01        4.0
2024-09-01        3.9
2024-10-01        4.1
2024-11-01        4.3
2024-12-01        4.4
2025-01-01        4.4
2025-02-01        3.2
2025-03-01        3.5
2025-04-01        3.3
2025-05-01        3.0
2025-06-01        2.8
2025-07-01        2.9
2025-08-01        2.7
2025-09-01        2.7
2025-10-01        2.4
2025-11-01        2.4
2025-12-01        2.3


In [29]:
print("=== Data Summary ===")
print(f"\n✓ European indices (yfinance): {close.shape[0]} daily points from {close.index[0].date()} to {close.index[-1].date()}")
print(f"  Columns: {list(close.columns)}")

print(f"\n✓ ECB deposit rate (FRED): daily data, currently at {ecb_rate.iloc[-1]}%")

print(f"\n✓ Euro area inflation (Eurostat): {len(hicp_ea)} monthly points")
print(f"  From {hicp_ea.index[0].date()} to {hicp_ea.index[-1].date()}")
print(f"  Latest inflation: {hicp_ea['inflation'].iloc[-1]}%")

=== Data Summary ===

✓ European indices (yfinance): 514 daily points from 2024-05-22 to 2026-05-25
  Columns: ['CAC40', 'DAX', 'FTSE100', 'EuroStoxx50']

✓ ECB deposit rate (FRED): daily data, currently at 2.0%

✓ Euro area inflation (Eurostat): 24 monthly points
  From 2024-01-01 to 2025-12-01
  Latest inflation: 2.3%


In [1]:
import yfinance as yf
from data import INDICES

data = yf.download(
    tickers=list(INDICES.values()),
    period="2y",
    interval="1d",
    auto_adjust=True,
    progress=False
)

print("Columns structure:")
print(data.columns)
print("\nClose columns:")
print(data["Close"].columns.tolist())
print("\nSample data:")
print(data["Close"].tail(3))

DEBUG - API Key loaded: True
Columns structure:
MultiIndex([( 'Close',     '^FCHI'),
            ( 'Close',     '^FTSE'),
            ( 'Close',    '^GDAXI'),
            ( 'Close', '^STOXX50E'),
            (  'High',     '^FCHI'),
            (  'High',     '^FTSE'),
            (  'High',    '^GDAXI'),
            (  'High', '^STOXX50E'),
            (   'Low',     '^FCHI'),
            (   'Low',     '^FTSE'),
            (   'Low',    '^GDAXI'),
            (   'Low', '^STOXX50E'),
            (  'Open',     '^FCHI'),
            (  'Open',     '^FTSE'),
            (  'Open',    '^GDAXI'),
            (  'Open', '^STOXX50E'),
            ('Volume',     '^FCHI'),
            ('Volume',     '^FTSE'),
            ('Volume',    '^GDAXI'),
            ('Volume', '^STOXX50E')],
           names=['Price', 'Ticker'])

Close columns:
['^FCHI', '^FTSE', '^GDAXI', '^STOXX50E']

Sample data:
Ticker            ^FCHI         ^FTSE        ^GDAXI    ^STOXX50E
2026-05-21  8086.000000  10443.50000

In [6]:
from data import get_indices

close_2y = get_indices("2y")
print("Columns:", close_2y.columns.tolist())
print("\nLast valid values:")
print(close_2y.apply(lambda col: col.dropna().iloc[-1]))

Columns: ['CAC40', 'DAX', 'FTSE100', 'EuroStoxx50']

Last valid values:
CAC40           8258.259766
DAX            10466.299805
FTSE100        25389.099609
EuroStoxx50     6136.660156
dtype: float64


In [8]:
import yfinance as yf

INDICES = {
    "CAC40": "^FCHI",
    "DAX": "^GDAXI",
    "FTSE100": "^FTSE",
    "EuroStoxx50": "^STOXX50E"
}

data = yf.download(
    tickers=list(INDICES.values()),
    period="2y",
    interval="1d",
    auto_adjust=True,
    progress=False
)

close = data["Close"]
print("Before rename:", close.columns.tolist())

ticker_to_name = {v: k for k, v in INDICES.items()}
print("Dict:", ticker_to_name)

close = close.rename(columns=ticker_to_name)
print("After rename:", close.columns.tolist())
print("\nLast valid values:")
print(close.apply(lambda col: col.dropna().iloc[-1]))

Before rename: ['^FCHI', '^FTSE', '^GDAXI', '^STOXX50E']
Dict: {'^FCHI': 'CAC40', '^GDAXI': 'DAX', '^FTSE': 'FTSE100', '^STOXX50E': 'EuroStoxx50'}
After rename: ['CAC40', 'FTSE100', 'DAX', 'EuroStoxx50']

Last valid values:
Ticker
CAC40           8258.259766
FTSE100        10466.299805
DAX            25389.099609
EuroStoxx50     6136.660156
dtype: float64
